# Task Dataset Build

Species-detector dataset: target species (XC/eBird) plus other-present species and AudioSet ambient (latter two pooled into `non_target`). Downloads are cached.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path

import pyrootutils

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator="pyproject.toml",
    pythonpath=True,
    dotenv=True,
)

# Parameters

In [3]:
from building.polychirp_training import task_slug

LAT, LON = -76.588305,39.305613 # greater bmore area
RADIUS_KM = 50

TARGET_SPECIES = [
    "Mareca americana",
    "Anas platyrhynchos",
    "Antrostomus carolinensis",
    "Coccyzus americanus",
    "Columba livia",
    "Rallus elegans",
    "Rallus limicola",
    "Charadrius vociferus",
    "Tringa semipalmata",
    "Larus smithsonianus",
    "Hydroprogne caspia",
    "Thalasseus maximus",
    "Coragyps atratus",
    "Accipiter cooperii",
    "Haliaeetus leucocephalus",
    "Buteo lineatus",
    "Aegolius acadicus",
    "Megascops asio",
    "Strix varia",
    "Melanerpes carolinus",
    "Sphyrapicus varius",
    "Leuconotopicus villosus",
    "Colaptes auratus",
    "Empidonax virescens",
    "Empidonax alnorum",
    "Myiarchus cinerascens",
    "Myiarchus crinitus",
    "Vireo olivaceus",
    "Vireo flavifrons",
    "Vireo griseus",
    "Cyanocitta cristata",
    "Corvus brachyrhynchos",
    "Corvus ossifragus",
    "Baeolophus bicolor",
    "Poecile carolinensis",
    "Tachycineta bicolor",
    "Progne subis",
    "Regulus satrapa",
    "Thryothorus ludovicianus",
    "Polioptila caerulea",
    "Sitta pusilla",
    "Sturnus vulgaris",
    "Sialia sialis",
    "Hylocichla mustelina",
    "Turdus migratorius",
    "Passer domesticus",
    "Anthus rubescens",
    "Ammodramus savannarum",
    "Spizella passerina",
    "Spizella pusilla",
    "Passerella iliaca",
    "Ammospiza maritima",
    "Centronyx henslowii",
    "Melospiza melodia",
    "Pipilo erythrophthalmus",
    "Icteria virens",
    "Icterus spurius",
    "Agelaius phoeniceus",
    "Euphagus carolinus",
    "Quiscalus quiscula",
    "Quiscalus major",
    "Seiurus aurocapilla",
    "Helmitheros vermivorum",
    "Parkesia motacilla",
    "Mniotilta varia",
    "Protonotaria citrea",
    "Geothlypis philadelphia",
    "Geothlypis formosa",
    "Geothlypis trichas",
    "Setophaga ruticilla",
    "Setophaga cerulea",
    "Setophaga americana",
    "Setophaga pinus",
    "Setophaga dominica",
    "Setophaga discolor",
    "Setophaga nigrescens",
    "Piranga rubra",
    "Cardinalis cardinalis",
    "Spiza americana",
    "Passerina caerulea",
    "Passerina cyanea",
    "Passerina ciris",
]

TARGET_CLIPS_PER_SPECIES = 5000
NON_TARGET_CLIPS_PER_SPECIES = 300

# Area discovery (matches 00_area_analysis.ipynb defaults)
MIN_AREA_RECORDINGS = 10
ANALYSIS_MAX_BIRDNET = 500
ANALYSIS_MIN_CONF = 0.85

# AudioSet ambient noise across about 60 FOCUS_CLASSES
AS_CLIPS_PER_CLASS = 250
AS_MAX_TOTAL_CLIPS = 15000

# Slug-named folders keep the (species, area) context visible on disk.
COLLECTION_NAME = task_slug(TARGET_SPECIES, LAT, LON, RADIUS_KM)
print(COLLECTION_NAME)

task_s82_-77_39_r50


In [5]:
%pip install -e

Note: you may need to restart the kernel to use updated packages.


C:\Users\annab\Downloads\EDL\birdshitterReal\EDL-PolyChirp\.venv\Scripts\python.exe: No module named pip


In [6]:
import sys
print("Python Path:", sys.executable)

try:
    import tensorflow
    print("TensorFlow Version:", tensorflow.__version__)
except ImportError as e:
    print("TensorFlow Import Error:", e)

try:
    import birdnetlib
    print("birdnetlib Version:", birdnetlib.__version__)
except ImportError as e:
    print("birdnetlib Import Error:", e)

Python Path: C:\Users\annab\Downloads\EDL\birdshitterReal\EDL-PolyChirp\.venv\Scripts\python.exe
TensorFlow Import Error: No module named 'tensorflow'
birdnetlib Import Error: No module named 'tensorflow'


# Discover other species present in the area

In [4]:
from building.analysis import (
    area_audio_cache_dir,
    bbox_from_radius,
    birdnet_area_analysis,
    combined_species_table,
    fetch_area_recordings,
    recordings_to_df,
)

bbox = bbox_from_radius(LAT, LON, RADIUS_KM)
area_df = recordings_to_df(await fetch_area_recordings(bbox))
bn = await birdnet_area_analysis(
    area_df,
    max_recordings=ANALYSIS_MAX_BIRDNET,
    min_confidence=ANALYSIS_MIN_CONF,
    cache_dir=area_audio_cache_dir(LAT, LON, RADIUS_KM),
)
combined = combined_species_table(
    area_df, bn["detections"], min_recordings=MIN_AREA_RECORDINGS
)
OTHER_SPECIES = [
    sp for sp in combined["scientific_name"].tolist() if sp not in TARGET_SPECIES
]
print(f"{len(OTHER_SPECIES)} other species present in area")
OTHER_SPECIES

C:\Users\annab\Downloads\EDL\birdshitterReal\EDL-PolyChirp\.venv\Lib\site-packages\pydub\utils.py:170: RuntimeWarning: Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work
  warn("Couldn't find ffmpeg or avconv - defaulting to ffmpeg, but may not work", RuntimeWarning)


ModuleNotFoundError: No module named 'tensorflow'

# Stream-download target species

In [ ]:
from building.data.download import download_and_process
from building.data.sources import XCSource, EBirdSource

with XCSource() as xc, EBirdSource() as eb:
    for sp in TARGET_SPECIES:
        await download_and_process(sp, TARGET_CLIPS_PER_SPECIES, [xc, eb])

# Stream-download other species (250 each, pooled as non-target)

In [ ]:
with XCSource() as xc, EBirdSource() as eb:
    for sp in OTHER_SPECIES:
        await download_and_process(sp, NON_TARGET_CLIPS_PER_SPECIES, [xc, eb])

# Boost `no_bird` via OTHER_SPECIES reprocessing

BirdNET mines a 3 s window into `non_target_empty/` from recordings with no detection >= `BIRDNET_THRESHOLD` (0.92). Each pass doubles the `OTHER_SPECIES` quota; stops at `NO_BIRD_CAP` or when sources run out.

In [ ]:
from building.data.download import (
    BIRDNET_NO_BIRD_DIR, NO_BIRD_CAP, SUBSAMPLES_DIR, download_and_process,
)
from building.data.sources import XCSource, EBirdSource

NO_BIRD_TARGET = NO_BIRD_CAP   # 20_000


def n_no_bird() -> int:
    return sum(1 for _ in BIRDNET_NO_BIRD_DIR.glob("*.wav")) if BIRDNET_NO_BIRD_DIR.exists() else 0


def total_other_clips() -> int:
    return sum(
        sum(1 for _ in (SUBSAMPLES_DIR / sp.replace(" ", "_")).glob("*.wav"))
        for sp in OTHER_SPECIES
        if (SUBSAMPLES_DIR / sp.replace(" ", "_")).exists()
    )


curr_nb, curr_ot = n_no_bird(), total_other_clips()
print(f"start  no_bird={curr_nb}/{NO_BIRD_TARGET}  other_total={curr_ot}")

quota = NON_TARGET_CLIPS_PER_SPECIES
done = False
while not done and curr_nb < NO_BIRD_TARGET:
    quota *= 2
    print(f"\n>>> boost OTHER_SPECIES quota -> {quota}/source")
    with XCSource() as xc, EBirdSource() as eb:
        for sp in OTHER_SPECIES:
            before = n_no_bird()
            await download_and_process(sp, quota, [xc, eb], verbose=False)
            after = n_no_bird()
            print(f"  {sp:<32s} no_bird {before:>5d} -> {after:>5d} (+{after - before})")
            if after >= NO_BIRD_TARGET:
                done = True
                break
    new_nb, new_ot = n_no_bird(), total_other_clips()
    print(
        f"after pass: no_bird={new_nb} (+{new_nb - curr_nb})  "
        f"other_total={new_ot} (+{new_ot - curr_ot})"
    )
    if new_ot == curr_ot:
        print("no new OTHER_SPECIES clips - sources exhausted, stopping")
        break
    curr_nb, curr_ot = new_nb, new_ot

print(f"\nend    no_bird={curr_nb}/{NO_BIRD_TARGET}")

# Stream-download AudioSet ambient clips

In [ ]:
from building.data.audioset import AudioSetConfig, stream_download_audioset_async

as_cfg = AudioSetConfig(
    clips_per_class=AS_CLIPS_PER_CLASS,
    max_total_clips=AS_MAX_TOTAL_CLIPS,
)
await stream_download_audioset_async(as_cfg)

# Pre-split summary

In [ ]:
from building.data.dataset import AUDIOSET_DIR, BIRDNET_NO_BIRD_DIR, SUBSAMPLES_DIR


def _count(folder):
    return len(list(folder.glob("*.wav"))) if folder.exists() else 0


print("Target classes:")
for sp in TARGET_SPECIES:
    print(f"  {sp}: {_count(SUBSAMPLES_DIR / sp.replace(' ', '_'))}")

as_total, as_classes = 0, 0
if AUDIOSET_DIR.exists():
    for d in sorted(AUDIOSET_DIR.iterdir()):
        if d.is_dir():
            n = _count(d)
            if n:
                as_total += n
                as_classes += 1
print(f"\nAudioSet: {as_total} clips across {as_classes} classes")

other_total, other_species_present = 0, 0
for sp in OTHER_SPECIES:
    n = _count(SUBSAMPLES_DIR / sp.replace(" ", "_"))
    if n:
        other_total += n
        other_species_present += 1
print(
    f"\nOther birds: {other_total} clips "
    f"across {other_species_present}/{len(OTHER_SPECIES)} species"
)

print(f"\nBirdNET no-bird clips: {_count(BIRDNET_NO_BIRD_DIR)}")

# Per-source RMS distribution (pre-split)

Raw clip levels per source, 2 dB bins, -70..0 dBFS.

In [ ]:
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from building.data.dataset import AUDIOSET_DIR, BIRDNET_NO_BIRD_DIR, SUBSAMPLES_DIR


def _rms_dbfs(x: np.ndarray) -> float:
    x = np.asarray(x, dtype=np.float32)
    rms = np.sqrt(np.mean(x**2))
    return float(20.0 * np.log10(max(rms, 1e-10)))


def _scan(files: list) -> np.ndarray:
    out: list[float] = []
    for f in tqdm(files, desc=f"scan ({len(files)})", leave=False):
        try:
            x, _ = sf.read(str(f), dtype="float32", always_2d=False)
            if x.ndim > 1:
                x = x.mean(axis=-1)
            out.append(_rms_dbfs(x))
        except Exception:
            pass
    return np.array(out)


groups: list[tuple[str, np.ndarray]] = []

# One panel per target species.
for sp in TARGET_SPECIES:
    folder = SUBSAMPLES_DIR / sp.replace(" ", "_")
    if folder.exists():
        groups.append((sp, _scan(sorted(folder.glob("*.wav")))))

# Other birds aggregated.
other_files: list = []
for sp in OTHER_SPECIES:
    folder = SUBSAMPLES_DIR / sp.replace(" ", "_")
    if folder.exists():
        other_files.extend(folder.glob("*.wav"))
groups.append((f"Other birds ({len(OTHER_SPECIES)} sp.)", _scan(other_files)))

# BirdNET no-bird clips.
if BIRDNET_NO_BIRD_DIR.exists():
    groups.append(
        ("BirdNET no-bird", _scan(sorted(BIRDNET_NO_BIRD_DIR.glob("*.wav"))))
    )

# AudioSet aggregated across focus classes.
if AUDIOSET_DIR.exists():
    as_files: list = []
    for d in sorted(AUDIOSET_DIR.iterdir()):
        if d.is_dir():
            as_files.extend(d.glob("*.wav"))
    groups.append(("AudioSet (all)", _scan(as_files)))

print(
    f"{'group':<32s} {'n':>5s}  {'q05':>6s} {'q50':>6s} {'q95':>6s}  "
    f"{'min':>6s} {'max':>6s}"
)
for label, a in groups:
    if len(a) == 0:
        continue
    print(
        f"{label:<32s} {len(a):>5d}  "
        f"{np.quantile(a, 0.05):>+6.1f} {np.median(a):>+6.1f} {np.quantile(a, 0.95):>+6.1f}  "
        f"{a.min():>+6.1f} {a.max():>+6.1f}"
    )

bins = np.arange(-70, 1, 2)
n = len(groups)
fig, axes = plt.subplots(n, 1, figsize=(10, max(2.0, 1.6 * n)), sharex=True)
if n == 1:
    axes = [axes]
for ax, (label, a) in zip(axes, groups):
    if len(a):
        ax.hist(a, bins=bins, color="steelblue", edgecolor="black", alpha=0.85)
        ax.axvline(
            np.median(a), color="red", lw=1, ls="--", label=f"median {np.median(a):+.1f}"
        )
        ax.legend(loc="upper left", fontsize=8, frameon=False)
    ax.set_ylabel(label, rotation=0, ha="right", va="center")
axes[-1].set_xlabel("RMS (dBFS)")
fig.suptitle("Per-source RMS distribution (all data, pre-split)")
plt.tight_layout()
plt.show()


# Assemble train/val/test

In [ ]:
from building.data.dataset import build_task_dataset

dataset_path = build_task_dataset(
    COLLECTION_NAME,
    target_species=TARGET_SPECIES,
    non_target_species=OTHER_SPECIES,
)
print(dataset_path)